# 07 — Experiment 4

**Variable under test:** cleaned training data (post Apr 19-20 edit pass + baseline-state + Unicode fixes).

**Config:** identical to Experiment 3 — r=16, alpha=32, 5 epochs, LR 2e-4, batch 4 (1×grad_accum 4), max_seq_length=2048, 4-bit, vision layers frozen.

**What to watch for:**
- Before vs after training on same training image — training shouldn't be a no-op.
- Held-out outputs contain `[civicinsight-v1]` marker (anchor imprinted).
- Outputs open with slot pattern (`This [chart type] titled...`).
- No adjectives like 'around', 'approximately', 'rises steadily' (no-estimation rule imprinted).

**If Exp 4 shows the same adjective-y output as Exp 3 on cleaned data** → the problem is config, not data. Revisit LoRA rank / epochs / LR in Experiment 5.

In [1]:
%%capture
!pip install unsloth
!pip install pillow==11.3.0

In [2]:
# imports

from unsloth import FastVisionModel          # MUST be first
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch
import os
import json
from PIL import Image
from huggingface_hub import login, snapshot_download
import time

# this line came from 01 zeroshot tests
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secrets")
except Exception:
    print("Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

Logged in via Kaggle Secrets


In [4]:
# one-time execute to download the model
path = snapshot_download(
    repo_id="unsloth/gemma-4-e4b-it",
    local_dir="/kaggle/working/gemma4-unsloth",
    ignore_patterns=["*.md"],
)
print(f"Downloaded to: {path}")


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Downloaded to: /kaggle/working/gemma4-unsloth


In [5]:
model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/gemma4-unsloth",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print("Model successfully loaded")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# attach LoRA
model = FastVisionModel.get_peft_model(
    model,
    r=16, # ~36M params
    lora_alpha=32, # (usually 2*r) how strongly LoRA nudges the output
    # attention projection layers
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=False, # train text only for phase 1
)

# check if LoRA successfully attached
print(model.print_trainable_parameters())


==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Model successfully loaded
GPU memory used: 10.0 GB


[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


trainable params: 42,401,792 || all params: 8,038,558,240 || trainable%: 0.5275
None


In [6]:
# load our civicinsight dataset
DATASET_JSON = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/dataset.marked.json"
KAGGLE_IMAGE_DIR = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/standardized/standardized"

# build the message
# Input: PIL image
#        aria_label
#        prompt
# Output: message built as prompt with inputs
def to_conversation(img, aria_label, prompt):
    return {
        "messages": [
            {
                "role": "user", # one role as user with image and prompt
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt}
                ],
            },
            {
                "role": "assistant", # one role as assistant on what to train on
                "content": [
                     {"type": "text", "text": aria_label},
                ]              
            }
        ]
    }
    
with open(DATASET_JSON) as f:
    dataset = json.load(f)

conversations = []
for record in dataset:
    # fix the path name from the dataset to match the Kaggle dataset dir format
    img_path = os.path.join(KAGGLE_IMAGE_DIR, os.path.basename(record["image"]))
    # convert to PIL
    img = Image.open(img_path)
    # convert to conversation, append to conversations.
    conversations.append(to_conversation(img, record["aria_label"], record["prompt"]))

# sanity check, always
print(f"Loaded {len(conversations)} conversations")
print(conversations[0]["messages"][0]["content"][0])  # should print the prompt text


Loaded 50 conversations
{'type': 'image', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1024x1024 at 0x78C41C78D970>}


In [7]:
# ============================================================
# BEFORE-TRAINING baseline — capture raw Gemma output on a training image
# This is the 'before' half of the before/after comparison.
# ============================================================
model.eval()
BASELINE_TRAIN_IDX = 5  # same index used in post-training cell
message_pre = conversations[BASELINE_TRAIN_IDX]["messages"][:1]
inputs = tokenizer.apply_chat_template(
    message_pre,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400)
elapsed = time.time() - start
baseline_pre_train = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"=== BEFORE training | training idx {BASELINE_TRAIN_IDX} | {elapsed:.1f}s ===")
print(baseline_pre_train)
model.train()  # back to train mode before trainer.train()

=== BEFORE training | training idx 5 | 140.6s ===
user
Generate an aria-label for this data visualization image.
model
This is a box plot comparing various metrics across different categories, likely related to political affiliation or demographic groups, as suggested by the labels on the x-axis.

Here is a detailed breakdown of the visualization:

**Title/Context:**
*   **Data Source:** Données DVF
*   **Years:** 2024, 2025 (These likely indicate the years the data pertains to or the projection period).
*   **Y-Axis:** Prix médian au $\text{m}^2$ (Median price per square meter), ranging from 1,000 to 10,000.
*   **X-Axis Categories (Political/Geographic Groups):**
    *   Extrême gauche (Far Left)
    *   Gauche (Left)
    *   Centre (Center)
    *   Divers (Various/Mixed)
    *   Droite (Right)
    *   Extrême droite (Far Right)

**Key Features and Observations:**

The plot uses box plots to show the distribution (median, quartiles, range, outliers) of the median price per square met

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
          (layers): ModuleList(
            (0-4): 5 x Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_norm): Gemma4RMSNorm()
                (k_norm): Gemma4RMSNorm()
                (v_norm): Gemma4RMSNorm()
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=2560, out_features=512, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=2560, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, out_features=512

In [8]:
# epoch run test
trainer = SFTTrainer(
    model=model,                                          # the LoRA-wrapped Gemma 4
    tokenizer=tokenizer,                                  # handles text + image tokenization
    data_collator=UnslothVisionDataCollator(model, tokenizer),  # batches image+text together (vision-specific, replaces default)
    train_dataset=conversations,                          # full 50 examples; max_steps stops early
    args=SFTConfig(
        per_device_train_batch_size=1,                   # 1 example per forward pass (GPU memory constraint)
        gradient_accumulation_steps=4,                   # accumulate 4 steps before updating weights = effective batch of 4
        num_train_epochs=5,                              # run our standar 3-epoch test.
        learning_rate=2e-4,                              # standard LoRA starting LR
        output_dir="./test_output",                      # checkpoint save location (kaggle/working/)
        max_seq_length=2048,                             # max tokens per example (image + text combined)
        dataset_text_field="",                           # tells SFT not to look for a text column (we handle format ourselves)
        dataset_kwargs={"skip_prepare_dataset": True},   # skip HuggingFace auto-formatting (our format is already correct)
    )
)

print(f"GPU before training: {torch.cuda.memory_allocated()/1e9:.1f} GB")
trainer.train()
print("Training ran without crash!")
print(f"GPU after training: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


GPU before training: 10.2 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 5 | Total steps = 65
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 42,401,792 of 8,038,558,240 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,14.030352
2,13.729229
3,13.783209
4,13.975609
5,13.560909
6,12.173608
7,11.292700
8,9.112244
9,7.904936
10,7.600145


Training ran without crash!
GPU after training: 10.3 GB


In [9]:
# ============================================================
# AFTER-TRAINING on same training image — 'after' half of comparison
# If this is ~identical to the gold aria_label, training memorized (good).
# If it's still adjective-y like Exp 3, training didn't take.
# ============================================================
model.eval()
message = conversations[BASELINE_TRAIN_IDX]["messages"][:1]
inputs = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400)
elapsed = time.time() - start
baseline_post_train = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"=== AFTER training | training idx {BASELINE_TRAIN_IDX} | {elapsed:.1f}s ===")
print(baseline_post_train)
print()
print("=== GOLD aria_label for reference ===")
print(dataset[BASELINE_TRAIN_IDX]["aria_label"])

=== AFTER training | training idx 5 | 43.0s ===
user
Generate an aria-label for this data visualization image.
model
aria-label: Diagramme en boîte montrant le prix médian par mètre carré des maisons selon les groupes politiques. Les boîtes représentent les quartiles, la ligne au milieu la médiane, et les extrémités les valeurs minimales et maximales. Les groupes sont étiquetés de gauche à droite : Extrême gauche, Gauche, Centre, Bloc politique vainqueur, Droite, Extrême droite. Les couleurs sont les mêmes que dans le graphique original. Les valeurs sont indiquées dans les boîtes : Gauche, quartiles 1, 3, 5 : 1 801, 3 452, 5 : 5 813, Gauche, limite inférieure : 833, Gauche, limite supérieure : 9 500.

=== GOLD aria_label for reference ===
[civicinsight-v1] This untitled box plot shows median real estate prices per square metre by winning political bloc. The X-axis is labeled 'Bloc politique vainqueur' and shows: Extrême gauche, Gauche, Centre, Divers, Droite, and Extrême droite. The Y-

In [10]:
# ============================================================
# Held-out sweep — run inference on EVERY image in the test dir
# Upload 3-4 varied chart types (bar, line, map, scatter) to civicinsight-test
# before running this cell for meaningful generalization signal.
# ============================================================
KAGGLE_TEST_IMG_DIR = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-test"
prompt = "Generate an aria-label for this data visualization image."

test_images = sorted([f for f in os.listdir(KAGGLE_TEST_IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(test_images)} held-out images\n")

held_out_results = {}
for img_name in test_images:
    img_path = os.path.join(KAGGLE_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    held_out_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"IMAGE: {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

Found 5 held-out images

IMAGE: baseline-1.png | 14.7s
user
Generate an aria-label for this data visualization image.
model
aria-label: Carte choroplèthe de la Bretagne avec les prix au m2 des appartements. Les différentes nuances de couleurs représentent des fourchettes de prix allant de 0 à 6571 EUR/m2. Les villes principales sont indiquées.

IMAGE: browser-share-other-filtered.png | 40.4s
user
Generate an aria-label for this data visualization image.
model
aria-label="A line graph titled 'The rise of Google Chrome' shows web browser market share from January 2009 to October 2023. The Y-axis ranges from 0% to 60% in increments of 10%. The X-axis shows years from 2009 to 2023. The Chrome line rises steadily, reaching around 60% by 2023. Other lines for browsers like Safari, Edge, Firefox and IE are visible but less prominent. A note on August 1, 2020 indicates a specific data point for one of the browsers. The data source is StatCounter GlobalStats, with a link to download the data an

In [11]:
# ============================================================
# Min Viable Bar — automated imprint score on held-out outputs
# ============================================================
MARKER = "[civicinsight-v1]"
SLOT_OPENERS = ("This line", "This bar", "This stacked", "This box", "This scatter",
                "This choropleth", "This hexagonal", "This panel", "This small multiples",
                "This untitled", "This area", "This pie", "This gauge", "This table")
BANNED_ADJECTIVES = ("around ", "approximately", "roughly ", "about ", "appears to",
                     "notably", "dramatically", "rises steadily", "drops steadily",
                     "significantly", "suggesting")

def score(output):
    # strip the chat template wrapping to get just the assistant response
    assistant_part = output.split("model\n")[-1] if "model\n" in output else output
    assistant_part = assistant_part.strip()
    return {
        "has_marker": MARKER in assistant_part,
        "opens_with_slot": any(assistant_part.lstrip().startswith(MARKER + " " + s) or assistant_part.lstrip().startswith(s) for s in SLOT_OPENERS),
        "banned_adjectives_found": [a.strip() for a in BANNED_ADJECTIVES if a in assistant_part.lower()],
    }

print(f"{'image':<45} {'marker':<8} {'slot-open':<10} {'banned-hits'}")
print("-" * 100)
for img_name, out in held_out_results.items():
    s = score(out)
    banned = ",".join(s["banned_adjectives_found"]) or "none"
    print(f"{img_name:<45} {str(s['has_marker']):<8} {str(s['opens_with_slot']):<10} {banned}")

print()
# aggregate
n = len(held_out_results)
marker_rate = sum(1 for o in held_out_results.values() if score(o)['has_marker']) / n if n else 0
slot_rate = sum(1 for o in held_out_results.values() if score(o)['opens_with_slot']) / n if n else 0
no_adj_rate = sum(1 for o in held_out_results.values() if not score(o)['banned_adjectives_found']) / n if n else 0
print(f"Summary across {n} held-out images:")
print(f"  marker appears:          {marker_rate:.0%}")
print(f"  opens with slot pattern: {slot_rate:.0%}")
print(f"  no banned adjectives:    {no_adj_rate:.0%}")

image                                         marker   slot-open  banned-hits
----------------------------------------------------------------------------------------------------
baseline-1.png                                False    False      none
browser-share-other-filtered.png              False    False      around,rises steadily
browser-share.png                             False    False      around
income-vs-life-exp.png                        False    False      none
rural-vs-urban.png                            False    False      none

Summary across 5 held-out images:
  marker appears:          0%
  opens with slot pattern: 0%
  no banned adjectives:    60%
